# ASSIGNMENT NLP – 5: Fine-Tuning DistilBERT for POS Tagging & Chunking
**Dataset:** CoNLL-2000 via NLTK (no HuggingFace loading-script issues)  
**Model:** DistilBERT (`distilbert-base-uncased`)  
**Framework:** Hugging Face Transformers + PyTorch

## Step 0: Install Required Libraries

In [1]:
!pip install transformers datasets seqeval evaluate accelerate torch -q
!pip install nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


## Step 1: Import Libraries

In [2]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')
print(f'PyTorch: {torch.__version__} | GPU: {torch.cuda.is_available()}')

✅ All libraries imported!
PyTorch: 2.10.0+cu128 | GPU: True


## Task 1: Dataset Selection (10%)
**Dataset:** CoNLL-2000 Chunking corpus (via NLTK — no loading script errors)  
Contains both **POS tags** and **Chunk tags** per token.  
Format per token: `(word, POS_tag, chunk_tag)`

In [3]:
# ── Task 1: Download CoNLL-2000 via NLTK ─────────────────────────────────────
nltk.download('conll2000', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import conll2000
from nltk import tree2conlltags

train_sents = conll2000.chunked_sents('train.txt')
test_sents  = conll2000.chunked_sents('test.txt')

print(f'✅ Dataset loaded!')
print(f'Training sentences : {len(train_sents)}')
print(f'Test sentences     : {len(test_sents)}')

# Preview one sentence
sample_iob = tree2conlltags(train_sents[0])
print('\nSample sentence (word, POS, Chunk):')
for item in sample_iob[:6]:
    print(' ', item)

✅ Dataset loaded!
Training sentences : 8936
Test sentences     : 2012

Sample sentence (word, POS, Chunk):
  ('Confidence', 'NN', 'B-NP')
  ('in', 'IN', 'B-PP')
  ('the', 'DT', 'B-NP')
  ('pound', 'NN', 'I-NP')
  ('is', 'VBZ', 'B-VP')
  ('widely', 'RB', 'I-VP')


In [4]:
# ── Build label vocabularies from entire corpus ────────────────────────────────
def sent_to_tags(chunked_sent):
    iob    = tree2conlltags(chunked_sent)
    words  = [t[0] for t in iob]
    pos    = [t[1] for t in iob]
    chunks = [t[2] for t in iob]
    return words, pos, chunks

all_pos, all_chunk = set(), set()
for sent in list(train_sents) + list(test_sents):
    _, p, c = sent_to_tags(sent)
    all_pos.update(p)
    all_chunk.update(c)

pos_label_names   = sorted(all_pos)
chunk_label_names = sorted(all_chunk)

pos_label2id   = {l: i for i, l in enumerate(pos_label_names)}
pos_id2label   = {i: l for i, l in enumerate(pos_label_names)}
chunk_label2id = {l: i for i, l in enumerate(chunk_label_names)}
chunk_id2label = {i: l for i, l in enumerate(chunk_label_names)}

print(f'POS  labels ({len(pos_label_names)})  : {pos_label_names}')
print(f'Chunk labels ({len(chunk_label_names)}): {chunk_label_names}')

POS  labels (44)  : ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']
Chunk labels (7): ['B-NP', 'B-PP', 'B-VP', 'I-NP', 'I-PP', 'I-VP', 'O']


In [5]:
# ── Convert corpus to plain lists ─────────────────────────────────────────────
def corpus_to_lists(sents):
    words_l, pos_l, chunk_l = [], [], []
    for sent in sents:
        w, p, c = sent_to_tags(sent)
        words_l.append(w)
        pos_l.append([pos_label2id[t]   for t in p])
        chunk_l.append([chunk_label2id[t] for t in c])
    return words_l, pos_l, chunk_l

tr_w, tr_p, tr_c = corpus_to_lists(train_sents)
te_w, te_p, te_c = corpus_to_lists(test_sents)

# 90/10 train-validation split
split = int(len(tr_w) * 0.9)
val_w, val_p, val_c   = tr_w[split:], tr_p[split:], tr_c[split:]
tr_w,  tr_p,  tr_c    = tr_w[:split], tr_p[:split], tr_c[:split]

print(f'Train: {len(tr_w)} | Val: {len(val_w)} | Test: {len(te_w)}')

Train: 8042 | Val: 894 | Test: 2012


## Task 2: Data Preprocessing (15%)
Tokenize with DistilBERT tokenizer and align labels to subword tokens.

In [6]:
# ── Task 2: Tokenizer ─────────────────────────────────────────────────────────
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

# ── Alignment demo ────────────────────────────────────────────────────────────
demo = tokenizer(['playing', 'football'], is_split_into_words=True)
print('Subword tokens :', tokenizer.convert_ids_to_tokens(demo['input_ids']))
print('Word IDs       :', demo.word_ids())
print('→ Only first subword of each word gets the real label; others get -100')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased
Subword tokens : ['[CLS]', 'playing', 'football', '[SEP]']
Word IDs       : [None, 0, 1, None]
→ Only first subword of each word gets the real label; others get -100


In [7]:
# ── Tokenize + align labels ────────────────────────────────────────────────────
def tokenize_and_align(words_list, labels_list):
    """
    Tokenizes word-split sentences and aligns integer label IDs.
    Subword continuations and special tokens receive -100 (ignored by loss).
    """
    tokenized = tokenizer(
        words_list,
        truncation=True,
        max_length=128,
        is_split_into_words=True
    )
    all_labels = []
    for i, label_ids in enumerate(labels_list):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev = [], None
        for wid in word_ids:
            if wid is None:          # [CLS] / [SEP]
                aligned.append(-100)
            elif wid != prev:        # first subword of word
                aligned.append(label_ids[wid])
            else:                    # continuation subword
                aligned.append(-100)
            prev = wid
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized

# ── Build HF DatasetDict ──────────────────────────────────────────────────────
def make_hf(w, p, c):
    return Dataset.from_dict({'tokens': w, 'pos_tags': p, 'chunk_tags': c})

raw = DatasetDict({
    'train':      make_hf(tr_w, tr_p, tr_c),
    'validation': make_hf(val_w, val_p, val_c),
    'test':       make_hf(te_w,  te_p,  te_c),
})

pos_tok = raw.map(
    lambda x: tokenize_and_align(x['tokens'], x['pos_tags']),
    batched=True, remove_columns=raw['train'].column_names
)
chunk_tok = raw.map(
    lambda x: tokenize_and_align(x['tokens'], x['chunk_tags']),
    batched=True, remove_columns=raw['train'].column_names
)

print('✅ Tokenized & label-aligned!')
print('Sample input_ids :', pos_tok['train'][0]['input_ids'][:8])
print('Sample labels    :', pos_tok['train'][0]['labels'][:8])

Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

✅ Tokenized & label-aligned!
Sample input_ids : [101, 7023, 1999, 1996, 9044, 2003, 4235, 3517]
Sample labels    : [-100, 18, 13, 10, 18, 38, 26, 36]


## Task 3: Model Setup (15%)

In [8]:
# ── Task 3: Load both models ───────────────────────────────────────────────────
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(pos_label_names),
    id2label=pos_id2label,
    label2id=pos_label2id,
    ignore_mismatched_sizes=True
)
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id,
    ignore_mismatched_sizes=True
)
params = sum(p.numel() for p in pos_model.parameters())
print(f'✅ POS   model: {len(pos_label_names)} labels | {params:,} parameters')
print(f'✅ Chunk model: {len(chunk_label_names)} labels')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ POS   model: 44 labels | 66,396,716 parameters
✅ Chunk model: 7 labels


## Task 4: Training (20%)

In [9]:
# ── Task 4: Training utilities ────────────────────────────────────────────────
seqeval = evaluate.load('seqeval')

def make_compute_metrics(label_names):
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        preds = np.argmax(logits, axis=-1)
        true_labels, true_preds = [], []
        for pred_row, label_row in zip(preds, labels):
            tl, tp = [], []
            for p, l in zip(pred_row, label_row):
                if l != -100:
                    tl.append(label_names[l])
                    tp.append(label_names[p])
            true_labels.append(tl)
            true_preds.append(tp)
        res = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            'precision': res['overall_precision'],
            'recall':    res['overall_recall'],
            'f1':        res['overall_f1'],
            'accuracy':  res['overall_accuracy'],
        }
    return compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def make_args(out_dir):
    return TrainingArguments(
        output_dir=out_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        logging_steps=100,
        report_to='none',
    )

print('✅ Trainer utilities ready!')

✅ Trainer utilities ready!


In [10]:
# ── Train POS Model ───────────────────────────────────────────────────────────
print('=' * 50)
print('TRAINING: POS TAGGING MODEL')
print('=' * 50)
pos_trainer = Trainer(
    model=pos_model,
    args=make_args('./pos_model'),
    train_dataset=pos_tok['train'],
    eval_dataset=pos_tok['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_names),
)
pos_trainer.train()
print('\n✅ POS training complete!')

TRAINING: POS TAGGING MODEL


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.150120,0.123384,0.954134,0.954965,0.954549,0.967444
2,0.101238,0.090306,0.965199,0.965759,0.965479,0.975535
3,0.076797,0.085182,0.965631,0.966920,0.966275,0.976154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ POS training complete!


In [11]:
# ── Train Chunking Model ──────────────────────────────────────────────────────
print('=' * 50)
print('TRAINING: CHUNKING MODEL')
print('=' * 50)
chunk_trainer = Trainer(
    model=chunk_model,
    args=make_args('./chunk_model'),
    train_dataset=chunk_tok['train'],
    eval_dataset=chunk_tok['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_names),
)
chunk_trainer.train()
print('\n✅ Chunking training complete!')

TRAINING: CHUNKING MODEL


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.098042,0.084119,0.966580,0.967886,0.967233,0.977915
2,0.067303,0.068044,0.973158,0.975889,0.974521,0.982532
3,0.054884,0.066078,0.973706,0.977551,0.975625,0.983246


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Chunking training complete!


## Task 5: Evaluation (15%)

In [12]:
# ── Task 5: Evaluate on Test Set ──────────────────────────────────────────────
from transformers.utils.notebook import NotebookProgressCallback

# 1. Safely remove the notebook callback that expects an active training table
for callback in pos_trainer.callback_handler.callbacks.copy():
    if isinstance(callback, NotebookProgressCallback):
        pos_trainer.remove_callback(callback)

for callback in chunk_trainer.callback_handler.callbacks.copy():
    if isinstance(callback, NotebookProgressCallback):
        chunk_trainer.remove_callback(callback)

# 2. Assign the test datasets
pos_trainer.eval_dataset   = pos_tok['test']
chunk_trainer.eval_dataset = chunk_tok['test']

# 3. Run evaluation (this will now work without crashing!)
pos_eval   = pos_trainer.evaluate()
chunk_eval = chunk_trainer.evaluate()

# 4. Print Results
print('=' * 50)
print('EVALUATION RESULTS (Test Set)')
print('=' * 50)
print(f"\n📊 POS Tagging:")
print(f"   Precision : {pos_eval['eval_precision']:.4f}")
print(f"   Recall    : {pos_eval['eval_recall']:.4f}")
print(f"   F1 Score  : {pos_eval['eval_f1']:.4f}")
print(f"   Accuracy  : {pos_eval['eval_accuracy']:.4f}")
print(f"\n📊 Chunking:")
print(f"   Precision : {chunk_eval['eval_precision']:.4f}")
print(f"   Recall    : {chunk_eval['eval_recall']:.4f}")
print(f"   F1 Score  : {chunk_eval['eval_f1']:.4f}")
print(f"   Accuracy  : {chunk_eval['eval_accuracy']:.4f}")

EVALUATION RESULTS (Test Set)

📊 POS Tagging:
   Precision : 0.9634
   Recall    : 0.9654
   F1 Score  : 0.9644
   Accuracy  : 0.9739

📊 Chunking:
   Precision : 0.9667
   Recall    : 0.9705
   F1 Score  : 0.9686
   Accuracy  : 0.9786


## Task 6: Inference (10%)

In [14]:
# ── Task 6: Inference on Custom Sentences ────────────────────────────────────
def predict_tags(sentence, model, label_names):
    """Predict tags for a raw sentence string."""
    model.eval()
    words  = sentence.split()

    # 1. Tokenize (this puts tensors on the CPU by default)
    inputs = tokenizer(
        words, is_split_into_words=True,
        return_tensors='pt', truncation=True, max_length=128
    )

    # Extract word_ids before we mess with the inputs dictionary
    word_ids = inputs.word_ids()

    # 2. THE FIX: Move the input tensors to the model's device (GPU)
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 3. Predict
    with torch.no_grad():
        logits = model(**inputs).logits

    preds = torch.argmax(logits, dim=-1)[0].tolist()

    # 4. Align and format results
    result, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        result.append((words[wid], label_names[preds[idx]]))
    return result

def show(sentence):
    pr = predict_tags(sentence, pos_model,   pos_label_names)
    cr = predict_tags(sentence, chunk_model, chunk_label_names)
    print(f'\nInput : {sentence}')
    print(f'{"─"*56}')
    print(f'{"Word":<20} {"POS Tag":<15} {"Chunk Tag":<15}')
    print(f'{"─"*56}')
    for (word, pos), (_, chunk) in zip(pr, cr):
        print(f'{word:<20} {pos:<15} {chunk:<15}')
    print(f'{"─"*56}')

for s in [
    'John works at Google in California',
    'The quick brown fox jumps over the lazy dog',
    'Apple is looking at buying a startup in the United Kingdom',
    'She sold seashells by the seashore'
]:
    show(s)


Input : John works at Google in California
────────────────────────────────────────────────────────
Word                 POS Tag         Chunk Tag      
────────────────────────────────────────────────────────
John                 NNP             B-NP           
works                VBZ             B-VP           
at                   IN              B-PP           
Google               NNP             B-NP           
in                   IN              B-PP           
California           NNP             B-NP           
────────────────────────────────────────────────────────

Input : The quick brown fox jumps over the lazy dog
────────────────────────────────────────────────────────
Word                 POS Tag         Chunk Tag      
────────────────────────────────────────────────────────
The                  DT              B-NP           
quick                NNP             I-NP           
brown                NNP             I-NP           
fox                  NNP           

## Task 7: Comparison (10%)

In [15]:
# ── Task 7: Comparison Table ──────────────────────────────────────────────────
rows = [
    ('Definition',    'Assigns grammatical role per word',     'Groups words into phrases'),
    ('Granularity',   'Word-level',                            'Phrase / span-level'),
    ('Output format', 'Single tag per word (NN, VBZ...)',      'BIO tag (B-NP, I-VP, O)'),
    ('Label count',   str(len(pos_label_names)),               str(len(chunk_label_names))),
    ('Complexity',    'Easy — local context',                  'Medium — span boundaries'),
    ('F1 Score',      f"{pos_eval['eval_f1']:.4f}",            f"{chunk_eval['eval_f1']:.4f}"),
    ('Use case',      'Grammar correction, parsing',           'IE, slot filling, NER'),
]
print(f'{"Aspect":<22} {"POS Tagging":<32} {"Chunking":<30}')
print('─' * 84)
for aspect, pv, cv in rows:
    print(f'{aspect:<22} {pv:<32} {cv:<30}')
print('\n📌 Chunking builds on POS: it groups POS-tagged words into higher-level phrases.')

Aspect                 POS Tagging                      Chunking                      
────────────────────────────────────────────────────────────────────────────────────
Definition             Assigns grammatical role per word Groups words into phrases     
Granularity            Word-level                       Phrase / span-level           
Output format          Single tag per word (NN, VBZ...) BIO tag (B-NP, I-VP, O)       
Label count            44                               7                             
Complexity             Easy — local context             Medium — span boundaries      
F1 Score               0.9644                           0.9686                        
Use case               Grammar correction, parsing      IE, slot filling, NER         

📌 Chunking builds on POS: it groups POS-tagged words into higher-level phrases.
